In [28]:
# =========================
# Cell 1: Imports & Setup
# =========================

import struct
import csv
from pathlib import Path

# --- Paths ---
LOGFILE_PATH = Path(
    "/Users/soni/Github/Digital-Detectives_Thesis/data/training/logfile/logfile raw/01-PE-LogFile"
)

OUTPUT_DIR = Path(
    "/Users/soni/Github/Digital-Detectives_Thesis/data/training/logfile/logfile parsed"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "01-PE-LogFile_offsets.csv"

# --- NTFS constants ---
RCRD_SIGNATURE = b"RCRD"


# =========================
# Cell 2: Sequential RCRD Parser
# =========================

def parse_rcrd_record(data: bytes, offset: int):
    """
    Parse a single RCRD record from the $LogFile buffer at given offset.
    Returns a dict or None if invalid.
    """
    if len(data) - offset < 24:  # minimum header length
        return None

    if data[offset:offset+4] != RCRD_SIGNATURE:
        return None

    try:
        # Header fields (based on NTFS RCRD structure)
        lsn = struct.unpack_from("<Q", data, offset + 8)[0]        # LSN
        record_length = struct.unpack_from("<H", data, offset + 4)[0]  # Record length
        redo_op = struct.unpack_from("<H", data, offset + 24)[0]
        undo_op = struct.unpack_from("<H", data, offset + 26)[0]
        redo_length = struct.unpack_from("<H", data, offset + 28)[0]
        undo_length = struct.unpack_from("<H", data, offset + 30)[0]
        attribute_offset = struct.unpack_from("<H", data, offset + 32)[0]
        redo_offset = struct.unpack_from("<H", data, offset + 36)[0]

        redo_buffer = data[offset + redo_offset : offset + redo_offset + redo_length]

        return {
            "lsn": lsn,
            "redo_op": redo_op,
            "undo_op": undo_op,
            "attribute_offset": attribute_offset,
            "record_offset": offset,
            "redo_length": redo_length,
            "undo_length": undo_length,
            "redo_buffer": redo_buffer,
            "record_length": record_length
        }
    except Exception:
        return None


# =========================
# Cell 3: Parse full $LogFile and output CSV
# =========================

with open(LOGFILE_PATH, "rb") as lf, open(OUTPUT_CSV, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)

    # --- CSV HEADER ---
    writer.writerow([
        "LSN",
        "Redo_OP_Hex",
        "Undo_OP_Hex",
        "Record_Offset_Hex",
        "Attribute_Offset_Hex",
        "Redo_Length",
        "Undo_Length"
    ])

    data = lf.read()
    offset = 0
    total_records = 0

    while offset < len(data):
        record = parse_rcrd_record(data, offset)
        if record:
            writer.writerow([
                record["lsn"],
                f"0x{record['redo_op']:x}",
                f"0x{record['undo_op']:x}",
                f"0x{record['record_offset']:x}",
                f"0x{record['attribute_offset']:x}",
                record["redo_length"],
                record["undo_length"]
            ])
            offset += record["record_length"]  # jump to next record using RecordLength
            total_records += 1
        else:
            offset += 8  # increment to search next RCRD signature

print(f"✓ Sequential $LogFile parsing complete → {OUTPUT_CSV}")
print(f"Total records parsed: {total_records}")


✓ Sequential $LogFile parsing complete → /Users/soni/Github/Digital-Detectives_Thesis/data/training/logfile/logfile parsed/01-PE-LogFile_offsets.csv
Total records parsed: 16382
